# Core Correctness Tests
Run exhaustive structural checks on small graphs, independent probability calculations, KL decomposition tests, and BIF network validation. After all tests pass, results/tests.json will be updated.

In [1]:
import json, sys, types
from pathlib import Path

def _load_core_notebook():
    notebook_path = Path("00_bn_core.ipynb")
    document = json.loads(notebook_path.read_text(encoding="utf-8"))
    module = types.ModuleType("bn_core")
    module.__file__ = str(notebook_path.resolve())
    sys.modules["bn_core"] = module
    for cell_index, cell in enumerate(document["cells"]):
        if cell.get("cell_type") == "code":
            source = "".join(cell.get("source", []))
            exec(compile(source, f"{notebook_path}#cell-{cell_index}", "exec"), module.__dict__)
    return module

bn_core = _load_core_notebook()


In [2]:
"""Independent correctness checks; run before the experiment suite."""
from pathlib import Path
import itertools
import json
import math
import numpy as np
from bn_core import *

ROOT = Path.cwd().resolve()


def enumerate_joint(bn):
    joint=np.zeros(tuple(bn.cards))
    for x in itertools.product(*[range(r) for r in bn.cards]):
        prob=1.
        for v in range(len(bn.names)):
            prob*=bn.cpts[v][tuple(x[u] for u in bn.parents[v])+(x[v],)]
        joint[x]=prob
    return joint


def main():
    rng=np.random.default_rng(9271)
    bn=Network(["A","B","C","D"],[["0","1"]]*4,
               [(),(),(0,1),(2,)],
               [np.array([.3,.7]),np.array([.6,.4]),
                rng.dirichlet([1,1],size=4).reshape(2,2,2),
                rng.dirichlet([1,1],size=2)])
    full=enumerate_joint(bn)
    comparisons=0
    for ev in ({},{2:1},{3:0},{0:1,3:0}):
        for qlen in range(1,len(bn.names)-len(ev)+1):
            for query in itertools.permutations([v for v in range(4) if v not in ev],qlen):
                actual=marginal(bn.factors(),query,bn.cards,ev)
                expected=np.zeros(tuple(bn.cards[v] for v in query))
                for x in itertools.product(range(2),repeat=4):
                    if all(x[v]==state for v,state in ev.items()):
                        expected[tuple(x[v] for v in query)]+=full[x]
                expected/=expected.sum()
                np.testing.assert_allclose(actual,expected,rtol=1e-12,atol=1e-14)
                comparisons+=1
    C={2}
    HR,_,_=fgcr(4,bn.edges,C,[0,1,2,3])
    parents=graph_parents({0,1,3},HR)
    X,_=sample_conditional(bn,1000,{2:1},rng,float(marginal(bn.factors(),(),bn.cards,{2:1},False)))
    fit=fit_tables(X,parents,bn.cards)
    true={v:true_family(bn,{2:1},v,pa) for v,pa in parents.items()}
    local_kl=cross_entropy(fit,true)-sum(t["entropy"] for t in true.values())
    p=marginal(bn.factors(),(0,1,3),bn.cards,{2:1})
    q=marginal(fitted_factors(fit,bn.cards),(0,1,3),bn.cards)
    full_kl=float((p*np.log(p/q)).sum())
    assert abs(local_kl-full_kl)<1e-12
    b=weighted_bounds(fit,true,len(X))
    assert b["empirical_bound"]>=0 and b["oracle_bound"]>=0
    # Robust simplex optimizer: compare to exhaustive feasible grid points.
    w=np.array([.2,.3,.5]); vals=np.array([2.,4.,1.]); eps=.1
    answer=robust_simplex_upper(w,vals,eps)
    for i in range(101):
        for j in range(101-i):
            z=np.array([i,j,100-i-j])/100
            if np.max(np.abs(z-w))<=eps+1e-12:
                assert z@vals<=answer+1e-12
    assert abs(answer-2.4)<1e-12
    # Empty parent cells, zeros in true CPTs, root factors.
    tiny=fit_tables(np.zeros((1,4),dtype=int),parents,bn.cards)
    assert np.isfinite(weighted_bounds(tiny,true,1)["empirical_bound"])
    cases=0
    for p in range(1,6):
        possible=list(itertools.combinations(range(p),2))
        for mask in range(1<<len(possible)):
            E={e for i,e in enumerate(possible) if mask>>i&1}
            for cmask in range(1<<p):
                C={i for i in range(p) if cmask>>i&1}
                beta=list(range(p))
                HR,H,_=fgcr(p,E,C,beta)
                alpha=[v for v in beta if v in C]+[v for v in beta if v not in C]
                assert H==ordered_oracle(p,E,set(),alpha)
                assert HR==ordered_oracle(p,E,C,[v for v in beta if v not in C])
                cases+=1
    # Parsers must recover published network cardinalities and normalized rows.
    for name,p,e in (("child",20,25),("alarm",37,46)):
        net=read_bif(ROOT/"data"/(name+".bif.gz"))
        assert len(net.names)==p and len(net.edges)==e
        for table in net.cpts:
            np.testing.assert_allclose(table.sum(axis=-1),1)
    report=dict(status="passed",variable_elimination_comparisons=comparisons,
                structural_cases=cases,structural_edge_equalities=2*cases,
                kl_decomposition_abs_difference=abs(local_kl-full_kl),
                robust_simplex_grid_check="passed",empty_cell_check="passed",
                bif_network_checks="passed")
    (ROOT/"results").mkdir(exist_ok=True)
    (ROOT/"results"/"tests.json").write_text(json.dumps(report,indent=2))
    print(json.dumps(report,indent=2))


In [3]:
main()


{
  "status": "passed",
  "variable_elimination_comparisons": 98,
  "structural_cases": 33866,
  "structural_edge_equalities": 67732,
  "kl_decomposition_abs_difference": 1.0408340855860843e-17,
  "robust_simplex_grid_check": "passed",
  "empty_cell_check": "passed",
  "bif_network_checks": "passed"
}
